In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn import RMSNorm, ModuleList
from torch.distributions import Categorical

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import time
import random
import json
import os
from tqdm.auto import tqdm
import math

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

## Comparison of forward pass speeds between minGRU and vanilla GRU

### Vanilla GRU

In [3]:
class vanilla_GRU(nn.Module):
    def __init__(self, input_features, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.W_r = nn.Linear(input_features + hidden_size, hidden_size)
        self.W_z = nn.Linear(input_features + hidden_size, hidden_size)
        self.W_h_tilda = nn.Linear(input_features + hidden_size, hidden_size)

    def forward(self, x, hidden):
        r_t = torch.sigmoid(self.W_r(torch.cat((x, hidden), dim=1)))
        z_t = torch.sigmoid(self.W_z(torch.cat((x, hidden), dim=1)))
        h_tilda = torch.tanh(self.W_h_tilda(torch.cat((x, r_t * hidden), dim=1)))
        h_t = (1 - z_t) * hidden + (z_t * h_tilda)
        return h_t

### minGRU

+ The paper discusses that in RNNs, state expansion is often used (i.e., $d_h = \alpha d_x$, where $\alpha \geq 1$), which helps the models better capture features from the input data.
<br><br>
+ We use a projection layer at the end to reduce the expanded dimension back to the original input size.
<br><br>

In [193]:
def g(x):
    return torch.where(x >= 0, x + 0.5, x.sigmoid())

def log_g(x):
    return torch.where(x >= 0, (F.relu(x) + 0.5).log(), -F.softplus(-x))

def parallel_scan_log(log_coeffs, log_values):
    a_star = F.pad(torch.cumsum(log_coeffs, dim=1), (0, 0, 1, 0))
    log_h0_plus_b_star = torch.logcumsumexp(log_values - a_star, dim=1)
    log_h = a_star + log_h0_plus_b_star
    return torch.exp(log_h)[:, 1:]
    

class minGRU(nn.Module):
    def __init__(self, input_features, expansion_factor=2):
        super().__init__()
        self.linear_z = nn.Linear(input_features, input_features * expansion_factor, bias=False)
        self.linear_h = nn.Linear(input_features, input_features * expansion_factor, bias=False)
        self.output_projection = nn.Linear(input_features * expansion_factor, input_features, bias=False)

    def forward(self, x, h0):
        seq_len = x.shape[1]
        update_gate = self.linear_z(x)
        hidden_state = self.linear_h(x)
        k = -F.softplus(update_gate)
        log_z = -F.softplus(-k)
        log_coeffs = -F.softplus(k)
        log_h_0 = log_g(h_0)
        log_tilde_h = log_g(hidden_state)        
        log_values = torch.cat([log_h_0, log_z + log_tilde_h], dim=1)
        output = parallel_scan_log(log_coeffs, log_values)
        output = output[:, -x.shape[1]:]
        latest_hidden = output[:, -1:]
        output = self.output_projection(output)
        return output, latest_hidden

    def sequential_mode(self, x_t, h_prev):
        z_t = torch.sigmoid(self.linear_z(x_t))
        h_tilde = self.linear_h(x_t)
        h_t = (1 - z_t) * h_prev + z_t * h_tilde
        return self.output_projection(h_t), h_t

### Hyperparameters

In [124]:
batch_size = 64
seq_len = 512
input_features = 256
hidden_size = 512

### Forward pass for vanilla GRU

In [127]:
x = torch.rand(batch_size, seq_len, input_features, device=device)
hidden_state = torch.zeros(batch_size, hidden_size, device=device)
model = vanilla_GRU(input_features=input_features, hidden_size=hidden_size).to(device)

model(x[:, 0, :], hidden_state)
num_runs = 30
start_time = time.time()
with torch.no_grad():
    for _ in tqdm(range(num_runs)):
        for t in range(seq_len):
            hidden_state = model(x[:, t, :], hidden_state)
    end_time = time.time()
    average_runtime_ms = ((end_time - start_time) / num_runs) * 1000
    print(f"Average runtime per timestep over {num_runs} runs: {average_runtime_ms:.2f} ms")

  0%|          | 0/30 [00:00<?, ?it/s]

Average runtime per timestep over 30 runs: 147.21 ms


### Forward pass for minGRU

In [128]:
model = minGRU(input_features=256).to(device)
expansion_factor=2
x = torch.rand(batch_size, seq_len, input_features, device=device)
h_0 = torch.zeros(batch_size, 1, input_features * expansion_factor, device=device)


model(x, h_0)

num_runs = 30
start_time = time.time()
with torch.no_grad():
    for _ in tqdm(range(num_runs)):
        output, _ = model(x, h_0)        
    end_time = time.time()
    average_runtime_ms = ((end_time - start_time) / num_runs) * 1000
    print(f"Average runtime per timestep over {num_runs} runs: {average_runtime_ms:.2f} ms")

  0%|          | 0/30 [00:00<?, ?it/s]

Average runtime per timestep over 30 runs: 2.06 ms


# Language modelling

### Loading and encoding the dataset (Shakespeare.txt)

In [49]:
with open('/kaggle/input/shakespeare-data/shakespeare.txt', 'r') as f:
    data = f.read()

chars = sorted(list(set(data)))
data_size, vocab_size = len(data), len(chars)

print("----------------------------------------")
print(f"Data has {data_size} characters, {vocab_size} unique")
print("----------------------------------------")

char_to_idx = { ch:i for i,ch in enumerate(chars) }
idx_to_char = { i:ch for i,ch in enumerate(chars) }

data = list(data)
for i, ch in enumerate(data):
    data[i] = char_to_idx[ch]

data = torch.tensor(data).to(device)

----------------------------------------
Data has 1115394 characters, 65 unique
----------------------------------------


### Function for generating batches of data

In [50]:
def get_batches(data, batch_size, seq_length):
    total_length = data.size(0)
    num_batches = (total_length - 1) // (batch_size * seq_length)
    data = data[:num_batches * batch_size * seq_length]
    data = data.view(batch_size, -1)
    for i in range(0, data.size(1) - seq_length, seq_length):
        x = data[:, i:i + seq_length]
        y = data[:, i + 1:i + seq_length + 1]
        yield x, y

### Convolution layer and MLP

+ The paper  discusses prepending a convolution layer with a kernel size of 4 before the recurrent unit and adding an MLP layer after it, as defined below.

#### Convolution layer
+ groups=`input_dim` means that each input channel is processed separately, and no information is shared across channels.
<br><br>
+ The padding is used to apply causal convolutions, where each output depends only on previous inputs (no future context).
<br><br>

In [51]:
class CausalConvlution_1D(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv1d(in_channels=input_dim, out_channels=input_dim,
                      kernel_size=4,
                      groups=input_dim)
        )
        
    def forward(self, x):
        x = x.transpose(1, 2)
        
        x = F.pad(input=x,
                  pad=(3, 0),
                  value=0.)
        
        x = self.conv_layer(x)
        x = x.transpose(1, 2)
        return x

### MLP

In [27]:
def MLP(input_dim, MLP_expansion_factor):
    linear_layer_1 = nn.Linear(in_features=input_dim,
                               out_features=input_dim * MLP_expansion_factor)
    
    activation_layer = nn.GELU()
    
    linear_layer_2 = nn.Linear(in_features=MLP_expansion_factor * input_dim,
                               out_features=input_dim)
    
    return nn.Sequential(linear_layer_1, activation_layer, linear_layer_2)

## Language Model

In [224]:
class LanguageModel(nn.Module):
    def __init__(self,
                 batch_size,
                 vocab_size,
                 input_features,
                 num_layers,
                 MLP_expansion_factor=4,
                 min_gru_expansion=2):
        
        super(LanguageModel, self).__init__()
        self.token_emb = nn.Embedding(vocab_size, input_features)
        self.layers = ModuleList([])
        self.dropout = nn.Dropout(0.2)

        for _ in range(num_layers):
            self.layers.append(
                ModuleList([
                    CausalConvlution_1D(input_features),
                    RMSNorm(input_features),
                    minGRU(input_features=input_features),
                    RMSNorm(input_features),
                    MLP(input_features, MLP_expansion_factor)]
            ))

        self.norm = RMSNorm(input_features)
        self.to_logits = nn.Linear(input_features, vocab_size, bias = False)
        self.h_0 = torch.zeros(batch_size, 1, input_features * expansion_factor, device=device)

    def forward(self, x):
        x = self.token_emb(x)        
        for conv_layer, layer_norm_1, mingru, layer_norm_2, MLP in self.layers:
            x = conv_layer(x) + x
            x = layer_norm_1(x)
            output, h_0 = mingru(x, self.h_0)
            x = x + output
            x = layer_norm_2(x)
            x = MLP(x) + x
            
            x = self.dropout(x) + x

            self.h_0 = h_0
            
        embed = self.norm(x)
        logits = self.to_logits(embed)
        
        return logits

## Training Loop

In [143]:
def train(model, loss_fn, optimizer, seq_len, batch_size, epochs,
          train_data, test_data, save=False):

    train_data = train_data
    test_data = test_data
    train_data_size = train_data.shape[0]
    test_data_size = test_data.shape[0]

    training_losses = []
    testing_losses = []

    start_time = time.time()

    for i_epoch in tqdm(range(1, epochs + 1), desc="Training Epochs", unit="epoch"):

        model.train()
        running_loss = 0
        n = 0

        for input_seq, target_seq in get_batches(train_data, batch_size, seq_len):

            output = model(input_seq)

            loss = loss_fn(output.view(-1, vocab_size),
                           target_seq.reshape(seq_len * batch_size))

            running_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            n += 1

        avg_train_loss = running_loss / n
        training_losses.append(avg_train_loss)

        model.eval()

        with torch.no_grad():
            test_running_loss = 0
            n_test = 0

            for test_input_seq, test_target_seq in get_batches(test_data,
                                                               batch_size, seq_len):

                test_output = model(test_input_seq)

                test_loss = loss_fn(test_output.view(-1, vocab_size),
                                    test_target_seq.reshape(seq_len * batch_size))

                test_running_loss += test_loss.item()

                n_test += 1

            avg_test_loss = test_running_loss / n_test
            testing_losses.append(avg_test_loss)

            print(f"Epoch: {i_epoch} \t Training Loss: {avg_train_loss:.4f} \t Testing Loss: {avg_test_loss:.4f}")
    end_time = time.time()
    training_time = end_time - start_time
    print("\nTraining Completed")
    print(f"Epochs Run: {epochs}")
    print(f"Time Taken: {round(training_time / 60, 2)} minutes")
    print(f"Time Taken per Epoch: {round((training_time) / epochs, 2)} seconds")

In [229]:
SEQ_LENGTH = 512
BATCH_SIZE = 64

LEARNING_RATE = 0.002
EPOCHS = 15

training_split = 0.90
train_data = data[:int(len(data) * training_split)].to(device)
test_data = data[int(len(data) * training_split):].to(device)

model = LanguageModel(batch_size=BATCH_SIZE,
                      vocab_size=vocab_size, input_features=256, num_layers=3)
model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [230]:
train(model=model, loss_fn=loss_fn, optimizer=optimizer, seq_len=SEQ_LENGTH,
      batch_size=BATCH_SIZE, epochs=EPOCHS, train_data=train_data,
      test_data=test_data)

Training Epochs:   0%|          | 0/15 [00:00<?, ?epoch/s]

Epoch: 1 	 Training Loss: 3.4290 	 Testing Loss: 3.2872
Epoch: 2 	 Training Loss: 2.9248 	 Testing Loss: 2.4969
Epoch: 3 	 Training Loss: 2.3460 	 Testing Loss: 2.2093
Epoch: 4 	 Training Loss: 2.0927 	 Testing Loss: 2.0205
Epoch: 5 	 Training Loss: 1.8922 	 Testing Loss: 1.9406
Epoch: 6 	 Training Loss: 1.7560 	 Testing Loss: 1.8553
Epoch: 7 	 Training Loss: 1.6452 	 Testing Loss: 1.8019
Epoch: 8 	 Training Loss: 1.5694 	 Testing Loss: 1.7696
Epoch: 9 	 Training Loss: 1.5096 	 Testing Loss: 1.7264
Epoch: 10 	 Training Loss: 1.4665 	 Testing Loss: 1.6942
Epoch: 11 	 Training Loss: 1.4261 	 Testing Loss: 1.6689
Epoch: 12 	 Training Loss: 1.3951 	 Testing Loss: 1.6427
Epoch: 13 	 Training Loss: 1.3713 	 Testing Loss: 1.6242
Epoch: 14 	 Training Loss: 1.3461 	 Testing Loss: 1.6256
Epoch: 15 	 Training Loss: 1.3260 	 Testing Loss: 1.6067

Training Completed
Epochs Run: 15
Time Taken: 2.71 minutes
Time Taken per Epoch: 10.84 seconds


### Text generation

In [260]:
text_length = 200

start_token = torch.randint(0, 65, (64, 1), dtype=torch.int64).to(device)

for _ in range(text_length):
    logits = model(start_token)
    logits = logits[:, -1]        
    temperature = 0.7
    scaled_logits = logits / temperature
    probabilities = torch.softmax(scaled_logits, dim=-1)
    sample = torch.multinomial(probabilities, num_samples=1)
    start_token = torch.cat((start_token, sample), dim = -1)

In [265]:
flat_list = start_token.flatten().tolist()
text = [idx_to_char[idx] for idx in flat_list]
text = ''.join(text)
print(text)

; if I cannot a mother.

SICINIUS:
Then, if you can better this?

CORIOLANUS:
You have you have thought the field.

LADY CAPULET:
Where is not peace to my mother would not holy too:
Thou dost art all tple you shall not spile in the foot.

GLOUCESTER:
No.

CORIOLANUS:
My lord, and but say your love as too soothe die,
The dear a post than thanks they be his sight:
'Tis ready and notice to him; he's a ke compense me, I know thee doth my father's
day be in this bear of themselves at my crown,
His deathmone dangerous will would be
To be no more. The mother may with it your cousin
And end revenge to thstand will strew than the war.

CORIOLANUS:
The more will not all strange it,--

Both Camillo,
For your grace and his gallant sleepers;
But with a senators it in the prince
Than with the hungry of thy with the gods good queen of my head.

GLOUCESTER:
What consul, my lord,
I'll say and then discover hath droppering.

HASTINGS:
Nor in God?

MERCUTIO:
Then take a word of the man forget:
By this wo